## Week 2: Vector Search — Part 1: Brute Force with Cosine Similarity

We encode text into vectors using a sentence transformer model, then find the most similar documents using dot product (cosine similarity on normalized vectors).

### Load the embedding model

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)  # full list: https://sbert.net/docs/sentence_transformer/pretrained_models.html

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Basic example: encoding and comparing two texts

In [2]:
q1 = "can i join post start"
v1 = model.encode(q1)

d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

print("Similarity (related):", v1.dot(dv))

Similarity (related): 0.3046617


In [3]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

print("Similarity (unrelated):", v2.dot(dv))

Similarity (unrelated): 0.019730574


Key properties of vector search:
* Similar texts get similar vectors; dot product measures how similar.
* Dot product of normalized vectors = cosine similarity:
  * `cos(0°) = 1` — vectors are identical
  * `cos(90°) = 0` — vectors are independent
  * `cos(180°) = -1` — vectors are exact opposites

### Load FAQ data and create embeddings

In [4]:
from ingest import load_faq_data

documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1368 documents


In [5]:
from tqdm.auto import tqdm
import numpy as np

texts = [doc["question"] + " " + doc["answer"] for doc in documents]

vectors = []
batch_size = 50

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i : i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Embeddings matrix shape: {X.shape}")

  0%|          | 0/28 [00:00<?, ?it/s]

Embeddings matrix shape: (1368, 384)


### Brute force vector search

Compute dot product between the query vector and all document vectors at once using matrix multiplication, then rank by score.

In [6]:
query = "can i join post start"
query_vector = model.encode(query)

scores = X.dot(query_vector)

best_idx = np.argmax(scores)
print(f"Best match index: {best_idx}, score: {scores[best_idx]:.4f}")
print(documents[best_idx])

Best match index: 2, score: 0.4389
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}


In [7]:
# argsort returns indices sorted lowest→highest; take the last 5 for the top results
top5_indices = np.argsort(scores)[-5:]

for idx in top5_indices:
    print(f"Score: {scores[idx]:.4f}")
    print(documents[idx])
    print()

Score: 0.4045
{'id': 'c843c4c541', 'course': 'stock-markets-analytics-zoomcamp', 'section': 'Projects', 'question': 'Can I start working on my project right away?', 'answer': 'Yes - the instructor recommends starting immediately, since completing the project is the most important outcome of the course.'}

Score: 0.4140
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

Score: 0.4230
{'id': '2d8b16c2a0', 'course': 'mlops-